# GRAIL Stage 2a reranker — full build-out (Colab Pro+)

Scales the **no-MCS bi-encoder reranker** (gate verdict: GO, val 0.497@15 > generator 0.440) toward a publishable headline: larger training, full-val selection + TEST-once + GLORYx-37 cross-method, seeds, ablations.

**Key finding driving this run (GLORYx pool-oracle diagnosis):** the external GLORYx gap to SyGMa is **RANKING + POOL-SIZE, not coverage** — the rule pool's oracle@15 is 0.372 at `top_k=100/max_pool=80` but **0.499 (= SyGMa) at `top_k=200/max_pool=150`**. So a **bigger inference pool** is the single most impactful knob; this notebook uses `top_k=200 max_pool=150` for both training and eval.

**Runtime → GPU.** Training is fast on GPU; the *pool generation* is RDKit/CPU-bound (the slow part, ~10-12s/substrate at top_k=200), so use a Pro+ high-RAM/high-core runtime and let pools cache to Drive (they survive disconnects).

**Put these in Google Drive** (dataset + checkpoint are gitignored): the GRAIL repo (push the branch to GitHub or zip to Drive), `train.sdf val.sdf test.sdf` + `*_triples*.txt`, and `artifacts/full5000_priors/checkpoints/generator.pt`. Edit the **CONFIG** cell, then Run all.

In [ ]:
# 1) Mount Drive + CONFIG (edit these paths)
from google.colab import drive
drive.mount('/content/drive')

GITHUB_URL = ''        # e.g. 'https://github.com/<you>/GRAIL.git'  (leave '' to copy from Drive instead)
BRANCH     = 'claude/hungry-pasteur-25d746'
DRIVE_REPO = '/content/drive/MyDrive/GRAIL'                 # used only if GITHUB_URL == ''
DRIVE_DATA = '/content/drive/MyDrive/GRAIL_data'            # train.sdf/val.sdf/test.sdf + *_triples*.txt
DRIVE_CKPT = '/content/drive/MyDrive/GRAIL_ckpt'            # generator.pt (+ filter.pt) of full5000_priors
DRIVE_CACHE= '/content/drive/MyDrive/GRAIL_reranker_cache'  # pools cache (survives disconnects)
TOP_K, MAX_POOL = 200, 150   # the oracle-raising knob (oracle@15: 100/80 -> 0.372 ; 200/150 -> 0.499)
print('config set; pool', TOP_K, MAX_POOL)

In [ ]:
# 2) Get the repo + install
import subprocess
if GITHUB_URL:
    subprocess.run(['git','clone','--branch',BRANCH,'--depth','1',GITHUB_URL,'/content/GRAIL'], check=True)
else:
    subprocess.run(['cp','-r',DRIVE_REPO,'/content/GRAIL'], check=True)
%cd /content/GRAIL
!pip -q install 'numpy<2' rdkit torch torch_geometric 2>&1 | tail -2
!pip -q install -e . 2>&1 | tail -2
import numpy, torch; print('numpy', numpy.__version__, '| torch', torch.__version__, '| cuda', torch.cuda.is_available())

In [ ]:
# 3) Wire Drive data + checkpoint + cache into the repo's expected paths (symlinks)
import os
os.makedirs('grail_metabolism/data', exist_ok=True)
for f in ['train.sdf','val.sdf','test.sdf','train_triples.txt','val_triples.txt','test_triples.txt',
          'train_triples_clean.txt','val_triples_clean.txt','test_triples_clean.txt']:
    src = os.path.join(DRIVE_DATA, f)
    dst = os.path.join('grail_metabolism/data', f)
    if os.path.exists(src) and not os.path.exists(dst): os.symlink(src, dst)
os.makedirs('artifacts/full5000_priors/checkpoints', exist_ok=True)
for f in ['generator.pt','filter.pt']:
    src = os.path.join(DRIVE_CKPT, f)
    dst = os.path.join('artifacts/full5000_priors/checkpoints', f)
    if os.path.exists(src) and not os.path.exists(dst): os.symlink(src, dst)
os.makedirs(DRIVE_CACHE, exist_ok=True)
if not os.path.exists('artifacts/reranker_gate_cache'): os.symlink(DRIVE_CACHE, 'artifacts/reranker_gate_cache')
print('data:', sorted(os.listdir('grail_metabolism/data'))[:6])
print('ckpt:', os.listdir('artifacts/full5000_priors/checkpoints'))
!python -m pytest grail_metabolism/tests/test_reranker.py -q 2>&1 | tail -3

In [ ]:
# 4) SCALE training + multi-seed (no-MCS bi-encoder, bigger pool). Pool-gen is the wall
#    (~10-12s/substrate at top_k=200, CPU); it caches to Drive so a disconnect+rerun resumes.
import subprocess, json, glob
for seed in [0, 1]:
    subprocess.run(['python','scripts/run_reranker_gate.py','--arch','bi',
                    '--train-substrates','2500','--val-substrates','600',
                    '--top-k',str(TOP_K),'--max-pool',str(MAX_POOL),
                    '--epochs','20','--seed',str(seed)], check=True)
    subprocess.run(['cp','results/reranker_gate_bi.json',f'results/reranker_gate_bi_seed{seed}.json'], check=True)
print('--- per-seed val headlines ---')
for f in sorted(glob.glob('results/reranker_gate_bi_seed*.json')):
    h = json.load(open(f))['headline']; print(f, '-> reranker', round(h['reranker_recall@15'],3),
          '| gen', round(h['generator_alone_recall@15'],3), '| oracle', round(h['oracle_recall@15'],3))

In [ ]:
# 5) Cross-method headline: GLORYx-37 (+ clean TEST) at the BIGGER pool (the oracle-raising knob).
!python scripts/reranker_predict.py --substrates gloryx --top-k {TOP_K} --max-pool {MAX_POOL} \
    --out docs/benchmark/data/grail_reranker_gloryx.json
!python scripts/eval_on_gloryx.py --no-grail --no-sygma \
    --extra GRAIL_reranker=docs/benchmark/data/grail_reranker_gloryx.json \
            BioTransformer=docs/benchmark/data/biotransformer_gloryx.json \
            MetaPredictor=docs/benchmark/data/metapredictor_gloryx.json
# clean TEST (touch once for the final report):
!python scripts/reranker_predict.py --substrates test --top-k {TOP_K} --max-pool {MAX_POOL} \
    --out results/grail_reranker_test.json

In [ ]:
# 6) Ablations: quantify each scalar feature's contribution (flags exist).
#    Full bi-encoder uses rule_prior + gen_score features; drop each and compare val recall@15.
import subprocess, json
abl = {}
for name, extra in [('full', []), ('no_rule_prior', ['--no-rule-prior']), ('no_gen_score', ['--no-gen-score'])]:
    subprocess.run(['python','scripts/run_reranker_gate.py','--arch','bi',
                    '--train-substrates','2500','--val-substrates','600',
                    '--top-k',str(TOP_K),'--max-pool',str(MAX_POOL),'--epochs','20','--seed','0'] + extra, check=True)
    abl[name] = json.load(open('results/reranker_gate_bi.json'))['headline']['reranker_recall@15']
print('ablation val recall@15:', {k: round(v,3) for k,v in abl.items()})
# (RNS k-NN analogue-retrieval feature is a separate, larger component — deferred.)

In [ ]:
# 7) Download the results bundle
import shutil
shutil.make_archive('/content/reranker_results','zip','results')
from google.colab import files
files.download('/content/reranker_results.zip')